# CPTAC-3 Benchmark

This benchmark fetches level 3 DNA Methylation data from the CPTAC-3 project. It queries for the Illumina Methylation EPIC platform and primary tumour samples taken from solid tissue. Note that this does not fetch the healthy controls (yet).

## 1. Setup

### 1.1 Import Libraries

In [3]:
import sys
from pathlib import Path

# Add the `root/src` to the Python path to import the package
root = Path.cwd().parent
sys.path.append(str(root / 'src'))

In [56]:
from methyltrain.config.loader import load_config
from methyltrain.fs.layout import ProjectLayout
from methyltrain.utils.load_utils import save_audit_table

from methyltrain.api.steps import (
    download, 
    clean_data, 
    quality_control, 
    preprocess, 
    aggregate_cohort, 
    cohort_batch_correction,
    load_raw_project,
    load_processed_project,
    save_project
)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1.2 Configurations

In [5]:
# Load the CPTAC-3 configurations YAML as found in `config/`
config_path = "../config/CPTAC-3_config.yaml"
config = load_config(config_path)

In [6]:
# Initialize the default project layout
project = config.get('project_id', 'CPTAC-3')
layout = ProjectLayout(
    project_name  = project,
    root_dir      = f'../data/{project}_benchmark',
    raw_dir       = f'../data/{project}_benchmark/raw',
    audit_table   = f'../data/{project}_benchmark/{project}_audit_table.csv',
    metadata      = f'../data/{project}_benchmark/{project}_metadata.csv',
    manifest      = f'../data/{project}_benchmark/{project}_manifest.csv',
    status_log    = f'../data/{project}_benchmark/{project}_status_log.csv',
    project_adata = f'../data/{project}_benchmark/{project}_adata.h5ad'
)

# Initialize the project layout file-paths
layout.initialize()
layout.validate()

## 2. Dataset Preparation

In [7]:
# Download the methylation data and produce all metadata, returning an audit table
audit_table = download(config, layout, verbose = True)

=====| Beginning Project Data Download |=====

~~~~~| 1. Building the Manifest |~~~~~
Successfully queried for the manifest
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

~~~~~| 4. Downloading Methylation Data |~~~~~
Successfully downloaded the methylation data
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

~~~~~| 2. Building the Metadata |~~~~~
Successfully queried for batch 0
Successfully queried for batch 20
Successfully queried for batch 40
Successfully queried for batch 60
Successfully queried for batch 80
Successfully queried for batch 100
Successfully queried for batch 120
Successfully queried for batch 140
Successfully queried for batch 160
Successfully queried for batch 180
Successfully queried for batch 200
Successfully queried for batch 220
Successfully queried for batch 240
Successfully queried for batch 260
Successfully queried for batch 280
Successfully queried for batch 300
Successfully queried for batch 320
Successfully queried for batch 340
Successfully queried for batch 360


In [8]:
# Clean the data by converting all raw `.txt` files into `.parquet` and removing nested artifacts
audit_table = clean_data(audit_table, layout)

In [29]:
# Load the raw data as an AnnData object
adata = load_raw_project(config, layout)

Before quality control and preprocessing, we have 1387 samples and 866552 probes

In [30]:
adata

AnnData object with n_obs × n_vars = 1387 × 866552
    obs: 'file_name', 'data_type', 'data_category', 'experimental_strategy', 'platform', 'project_id', 'submitter_id', 'sample_type', 'aliquot_id', 'status'
    uns: 'project_id', 'platform', 'reference_genome', 'level', 'data_type', 'conversion', 'state', 'preprocessing_steps', 'split', 'split_percentage'

In [50]:
# Perform quality control on the data
adata, audit_table = methyltrain.quality_control(adata, audit_table, config, layout, verbose = True)
# adata, audit_table = quality_control(adata, audit_table, config, layout, verbose = True)

=====| Beginning Quality Control |=====
Loaded annotation for platform Illumina Methylation Epic and reference genome GRCh38
Successfully performed sample QC
Successfully performed probe QC
=====| Finished Quality Control |=====


After quality control, we have 924 samples and 543605 probes

In [51]:
adata

AnnData object with n_obs × n_vars = 924 × 543605
    obs: 'file_name', 'data_type', 'data_category', 'experimental_strategy', 'platform', 'project_id', 'submitter_id', 'sample_type', 'aliquot_id', 'status', 'missing_rate', 'mean'
    var: 'missing_rate'
    uns: 'project_id', 'platform', 'reference_genome', 'level', 'data_type', 'conversion', 'state', 'preprocessing_steps', 'split', 'split_percentage'

In [53]:
# Perform preprocessing on the data
adata = preprocess(adata, config)

After preprocessing, we have 924 samples and 450920 probes (includes filtering for low variance probes)

In [54]:
adata

AnnData object with n_obs × n_vars = 924 × 450920
    obs: 'file_name', 'data_type', 'data_category', 'experimental_strategy', 'platform', 'project_id', 'submitter_id', 'sample_type', 'aliquot_id', 'status', 'missing_rate', 'mean'
    var: 'missing_rate', 'variance', 'frac_imputed', 'impute_value'
    uns: 'project_id', 'platform', 'reference_genome', 'level', 'data_type', 'conversion', 'state', 'preprocessing_steps', 'split', 'split_percentage'

## 3. Dataset Saving

In [57]:
# Save both the project and the audit table
save_project(adata, layout)
save_audit_table(audit_table, layout)

## 4. Cleaning

In [ ]:
# Optional clean-up: delete raw data, reproducible as metadata persists
for parquet_file in layout.raw_dir.glob("*.parquet"):
    parquet_file.unlink()